# SEC EDGAR Gold Layer - Company Financials

Business-ready financial statements and KPIs built from `sec_edgar.silver.company_facts`, published to `sec_edgar.gold` (stored under the `gold_usa` external location) with a write-audit-publish pattern.

**Gold tables**
* `dim_company` - company attributes and fiscal year-end.
* `dim_metric` - governed KPI dictionary: business definition and ordered XBRL concept fallbacks per metric.
* `dim_fiscal_calendar` - per-company fiscal years derived from 10-K periods (handles Sep/Jun/Jan year-ends and 52/53-week years), plus the open current year.
* `fact_financials_annual` - one row per company / fiscal year / metric, as currently reported (latest filing), with the originally reported value and a restatement flag.
* `fact_financials_quarterly` - one row per company / fiscal year / fiscal quarter / metric with discrete quarters (YTD cash flows and Q4 are derived and labelled).
* `company_financials_annual`, `company_financials_quarterly` - wide KPI tables (margins, FCF, growth, ROE/ROA, leverage, TTM).
* `audit_runs`, `dq_results` - run log and data quality evidence.

**Key business rules (validated against the data)**
* Concepts change over time (Apple/Microsoft `SalesRevenueNet` -> `RevenueFromContractWithCustomer...`); Walmart reports both and only `Revenues` includes membership income; `SalesRevenueGoodsNet` is goods-only -> ordered priority list per metric, resolved per company-period, concept used is stored for lineage.
* Fiscal year = year of fiscal year end (minus 7 days for 52/53-week years ending in early January), matching company naming (NVIDIA FY2024 ends Jan-2024).
* Discrete quarter = reported 3-month value, else YTD(n) - YTD(n-1); Q4 = FY - 9M YTD. Per-share and share-count metrics are non-additive and never derived.
* Silver must have a successful last run (gate); the silver snapshot version consumed is recorded for reproducibility via time travel.
* Gold is rebuilt deterministically from the pinned silver snapshot and only published when all ERROR-level checks (incl. known published figures) pass.

In [0]:
# Step 1: parameters, run context, dependency gate on silver, pinned silver snapshot (time travel) for reproducibility
import uuid, datetime, json
from pyspark.sql import functions as F, types as T, Window

dbutils.widgets.text("catalog", "sec_edgar")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("gold_location", "abfss://gold@stdsecusa.dfs.core.windows.net/SEC_EDGAR")
CAT      = dbutils.widgets.get("catalog")
SIL      = f"{CAT}.{dbutils.widgets.get('silver_schema')}"
GLD      = f"{CAT}.{dbutils.widgets.get('gold_schema')}"
GOLD_LOC = dbutils.widgets.get("gold_location").rstrip("/")
RUN_ID   = str(uuid.uuid4())
RUN_USER = spark.sql("select current_user()").first()[0]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GLD} MANAGED LOCATION '{GOLD_LOC}' COMMENT 'SEC EDGAR gold layer: business-ready financial statements and KPIs'")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {GLD}.audit_runs (
  run_id STRING NOT NULL, run_start TIMESTAMP, run_end TIMESTAMP, status STRING, run_user STRING,
  silver_run_id STRING, silver_facts_version BIGINT, table_row_counts MAP<STRING, BIGINT>,
  dq_errors INT, dq_warnings INT, error_message STRING) COMMENT 'One row per gold build'""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {GLD}.dq_results (
  run_id STRING, check_name STRING, severity STRING, failed_rows BIGINT, checked_rows BIGINT, passed BOOLEAN,
  details STRING, checked_at TIMESTAMP) COMMENT 'Gold data quality results per run'""")

# runs interrupted mid-way stay RUNNING; schedule the job with max_concurrent_runs = 1
spark.sql(f"UPDATE {GLD}.audit_runs SET status = 'ABANDONED', run_end = current_timestamp() WHERE status = 'RUNNING' AND run_start < current_timestamp() - INTERVAL 1 MINUTE")
spark.sql(f"INSERT INTO {GLD}.audit_runs (run_id, run_start, status, run_user) VALUES ('{RUN_ID}', current_timestamp(), 'RUNNING', '{RUN_USER}')")

def fail_run(msg):
    spark.sql(f"UPDATE {GLD}.audit_runs SET status = 'FAILED', run_end = current_timestamp(), error_message = {json.dumps(msg)} WHERE run_id = '{RUN_ID}'")
    raise Exception(msg)

# dependency gate: never build gold on top of a failed / partial silver load
last = spark.sql(f"SELECT run_id, status FROM {SIL}.audit_runs WHERE status NOT IN ('RUNNING', 'ABANDONED') ORDER BY run_start DESC LIMIT 1").first()
if last is None or last.status not in ("SUCCESS", "NO_CHANGES"):
    fail_run(f"Silver gate failed: last silver run status = {None if last is None else last.status}")
SILVER_RUN_ID  = last.run_id
SILVER_VERSION = spark.sql(f"DESCRIBE HISTORY {SIL}.company_facts LIMIT 1").first()["version"]
facts   = spark.sql(f"SELECT * FROM {SIL}.company_facts VERSION AS OF {SILVER_VERSION}").filter("is_active")
company = spark.table(f"{SIL}.company")
spark.sql(f"UPDATE {GLD}.audit_runs SET silver_run_id = '{SILVER_RUN_ID}', silver_facts_version = {SILVER_VERSION} WHERE run_id = '{RUN_ID}'")

# write-audit-publish: everything is staged first, checked, and published only if ERROR checks pass
STAGED = {}
def stage(df, name, keys, comment, cluster_by):
    stg = f"{GLD}._stg_{name}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(stg)
    STAGED[name] = dict(stg=stg, keys=keys, comment=comment, cluster_by=cluster_by)
    return spark.table(stg)

print(f"run_id={RUN_ID} silver_run={SILVER_RUN_ID} ({last.status}) silver_version={SILVER_VERSION} active_facts={facts.count():,}")

In [0]:
# Step 2: governed KPI dictionary. Concept order = priority (first available wins per company-period).
# (metric, label, statement, period_type, unit, is_additive, [concepts], definition)
METRICS = [
 ("revenue", "Total revenue", "income_statement", "duration", "USD", True,
  ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "RevenuesNetOfInterestExpense",
   "RevenueFromContractWithCustomerIncludingAssessedTax", "SalesRevenueGoodsNet"], "Total revenues; banks: net revenue after interest expense. SalesRevenueGoodsNet is last resort (goods only)."),
 ("cost_of_revenue", "Cost of revenue", "income_statement", "duration", "USD", True,
  ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"], "Cost of goods and services sold"),
 ("gross_profit", "Gross profit", "income_statement", "duration", "USD", True, ["GrossProfit"], "Gross profit as reported"),
 ("rnd_expense", "R&D expense", "income_statement", "duration", "USD", True, ["ResearchAndDevelopmentExpense"], "Research and development expense"),
 ("operating_income", "Operating income", "income_statement", "duration", "USD", True, ["OperatingIncomeLoss"], "Operating income (loss)"),
 ("interest_expense", "Interest expense", "income_statement", "duration", "USD", True, ["InterestExpense"], "Interest expense"),
 ("pretax_income", "Pre-tax income", "income_statement", "duration", "USD", True,
  ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
   "IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments"], "Income from continuing operations before income taxes"),
 ("income_tax_expense", "Income tax expense", "income_statement", "duration", "USD", True, ["IncomeTaxExpenseBenefit"], "Income tax expense (benefit)"),
 ("net_income", "Net income", "income_statement", "duration", "USD", True, ["NetIncomeLoss", "ProfitLoss"], "Net income attributable to the parent"),
 ("eps_basic", "EPS basic", "income_statement", "duration", "USD/shares", False, ["EarningsPerShareBasic"], "Basic earnings per share"),
 ("eps_diluted", "EPS diluted", "income_statement", "duration", "USD/shares", False, ["EarningsPerShareDiluted"], "Diluted earnings per share"),
 ("diluted_shares", "Diluted weighted avg shares", "income_statement", "duration", "shares", False, ["WeightedAverageNumberOfDilutedSharesOutstanding"], "Weighted average diluted shares outstanding"),
 ("dividends_per_share", "Dividends declared per share", "income_statement", "duration", "USD/shares", False, ["CommonStockDividendsPerShareDeclared"], "Dividends declared per common share"),
 ("operating_cash_flow", "Operating cash flow", "cash_flow", "duration", "USD", True,
  ["NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"], "Net cash from operating activities"),
 ("capex", "Capital expenditure", "cash_flow", "duration", "USD", True,
  ["PaymentsToAcquirePropertyPlantAndEquipment", "PaymentsToAcquireProductiveAssets"], "Purchases of property, plant and equipment (positive = cash out)"),
 ("total_assets", "Total assets", "balance_sheet", "instant", "USD", False, ["Assets"], "Total assets"),
 ("current_assets", "Current assets", "balance_sheet", "instant", "USD", False, ["AssetsCurrent"], "Total current assets"),
 ("total_liabilities", "Total liabilities", "balance_sheet", "instant", "USD", False, ["Liabilities"], "Total liabilities"),
 ("current_liabilities", "Current liabilities", "balance_sheet", "instant", "USD", False, ["LiabilitiesCurrent"], "Total current liabilities"),
 ("total_equity", "Stockholders equity", "balance_sheet", "instant", "USD", False,
  ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"], "Equity attributable to the parent"),
 ("liabilities_and_equity", "Liabilities and equity", "balance_sheet", "instant", "USD", False, ["LiabilitiesAndStockholdersEquity"], "Total liabilities and stockholders equity"),
 ("cash", "Cash and equivalents", "balance_sheet", "instant", "USD", False,
  ["CashAndCashEquivalentsAtCarryingValue", "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"], "Cash and cash equivalents"),
 ("long_term_debt", "Long-term debt", "balance_sheet", "instant", "USD", False, ["LongTermDebtNoncurrent", "LongTermDebt"], "Long-term debt (non-current preferred)"),
]
METRIC_NAMES = [m[0] for m in METRICS]
dim_metric_df = spark.createDataFrame(METRICS, "metric string, metric_label string, statement string, period_type string, unit string, is_additive boolean, concept_priority array<string>, definition string")
map_df = (dim_metric_df.select("metric", "unit", F.col("period_type").alias("metric_period_type"), "is_additive",
                               F.posexplode("concept_priority").alias("priority", "concept")))

# a concept may feed only one metric, otherwise values would be double counted
dup = map_df.groupBy("concept").count().filter("count > 1").count()
if dup: fail_run(f"{dup} concepts mapped to more than one metric")
stage(dim_metric_df, "dim_metric", ["metric"], "Governed KPI dictionary with XBRL concept priority", ["metric"])

# coverage of the dictionary in the pinned silver snapshot
display(map_df.join(facts.filter("taxonomy = 'us-gaap'").groupBy("concept", "unit").agg(F.countDistinct("cik").alias("companies"), F.count("*").alias("facts")), ["concept", "unit"], "left")
             .orderBy("metric", "priority"))

In [0]:
# Step 3: fiscal years from the data, not assumptions. Annual period = the (start, end) pair with most ~1-year 10-K facts.
annual_cand = (facts.filter("taxonomy = 'us-gaap' AND form IN ('10-K', '10-K/A') AND period_type = 'duration' AND duration_days BETWEEN 350 AND 380")
               .groupBy("cik", "period_start", "period_end").agg(F.count("*").alias("n_facts"))
               .withColumn("fiscal_year", F.year(F.date_sub("period_end", 7))))
w_fy = Window.partitionBy("cik", "fiscal_year").orderBy(F.desc("n_facts"), F.desc("period_end"))
closed = (annual_cand.withColumn("rn", F.row_number().over(w_fy)).filter("rn = 1 AND n_facts >= 10")
          .select("cik", "fiscal_year", F.col("period_start").alias("fy_start"), F.col("period_end").alias("fy_end"), "n_facts")
          .withColumn("is_closed", F.lit(True)))

# open (current) fiscal year: projected from the last closed year so in-progress 10-Q quarters are not lost
w_last = Window.partitionBy("cik").orderBy(F.desc("fiscal_year"))
last_fy = closed.withColumn("rn", F.row_number().over(w_last)).filter("rn = 1")
open_fy = last_fy.select("cik", (F.col("fiscal_year") + 1).alias("fiscal_year"), F.date_add("fy_end", 1).alias("fy_start"),
                         F.when(~(F.datediff("fy_end", "fy_start") + 1).isin(365, 366), F.date_add("fy_end", 364))
                          .otherwise(F.add_months("fy_end", 12)).alias("fy_end"),
                         F.lit(None).cast("bigint").alias("n_facts"), F.lit(False).alias("is_closed"))
cal = (closed.unionByName(open_fy)
       .withColumn("fy_days", F.datediff("fy_end", "fy_start") + 1)
       .withColumn("is_52_53_week", ~F.col("fy_days").isin(365, 366))
       .withColumn("fy_end_month", F.month("fy_end"))
       .withColumn("fiscal_year_label", F.concat(F.lit("FY"), F.col("fiscal_year").cast("string"))))
cal = stage(cal, "dim_fiscal_calendar", ["cik", "fiscal_year"], "Per-company fiscal calendar derived from 10-K periods; is_closed=false is the in-progress year", ["cik"])

dim_company = (company.select("cik", "cik_padded", "entity_name", "taxonomies", "fact_count", "last_filed_date")
   .join(cal.groupBy("cik").agg(F.min("fiscal_year").alias("first_fiscal_year"),
                                F.max(F.when(F.col("is_closed"), F.col("fiscal_year"))).alias("last_closed_fiscal_year"),
                                F.max_by("fy_end_month", "fiscal_year").alias("fiscal_year_end_month")), "cik", "left")
   .withColumn("sec_company_url", F.concat(F.lit("https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK="), F.col("cik_padded"))))
dim_company = stage(dim_company, "dim_company", ["cik"], "Company dimension", ["cik"])
display(dim_company)
display(cal.filter("fiscal_year >= 2023").orderBy("cik", "fiscal_year"))

In [0]:
# Step 4: keep only the 'as currently reported' value per period, attach fiscal year / quarter, classify the period shape
# Only financial-statement filings are authoritative. Proxy statements / prospectuses (DEF 14A, 424B, 8-K) re-tag rounded
# figures later (e.g. JPM FY2024 net income 58.5bn in a DEF 14A vs 58,471m in the 10-K), so silver's cross-form
# is_latest_filing is NOT used here: we re-rank the latest value among periodic forms only.
FS_FORMS = ["10-K", "10-K/A", "10-Q", "10-Q/A", "20-F", "20-F/A", "40-F", "40-F/A", "10-KT", "10-QT"]
w_latest_fs = Window.partitionBy("cik", "taxonomy", "concept", "unit", "period_start", "period_end").orderBy(F.desc("filed_date"), F.desc("accession_number"))
fs_facts = facts.filter(F.col("form").isin(FS_FORMS))
m = (fs_facts.filter("taxonomy = 'us-gaap'")
     .withColumn("_rn", F.row_number().over(w_latest_fs)).filter("_rn = 1").drop("_rn")
     .join(F.broadcast(map_df), ["concept", "unit"])
     .filter(F.col("period_type") == F.col("metric_period_type")))

c = cal.select("cik", "fiscal_year", "fy_start", "fy_end")
# a period belongs to the fiscal year whose window contains its end (60-day floor avoids double mapping at boundaries)
mc = (m.alias("m").join(F.broadcast(c).alias("c"),
        (F.col("m.cik") == F.col("c.cik")) & (F.col("m.period_end") > F.date_add(F.col("c.fy_start"), 60)) &
        (F.col("m.period_end") <= F.date_add(F.col("c.fy_end"), 10)))
      .select("m.*", "c.fiscal_year", "c.fy_start", "c.fy_end"))

mc = (mc.withColumn("fiscal_quarter", F.round(F.months_between("period_end", F.date_sub("fy_start", 1)) / 3).cast("int"))
        .withColumn("expected_end", F.when(F.col("fiscal_quarter") == 4, F.col("fy_end"))
                                     .otherwise(F.add_months(F.date_sub("fy_start", 1), F.col("fiscal_quarter") * 3)))
        .withColumn("is_aligned", F.col("fiscal_quarter").between(1, 4) & (F.abs(F.datediff("period_end", "expected_end")) <= 10))
        .withColumn("is_ytd", (F.col("period_type") == "duration") & (F.abs(F.datediff("period_start", "fy_start")) <= 10))
        .withColumn("is_disc", (F.col("period_type") == "duration") & F.col("duration_days").between(80, 100))
        .withColumn("is_inst", F.col("period_type") == "instant"))

mapped = mc.filter("is_aligned AND (is_inst OR is_ytd OR is_disc)")
total_m, total_mapped = m.count(), mapped.count()
print(f"metric facts (latest filing)={total_m:,}  mapped to a fiscal quarter-end={total_mapped:,}  not mapped={total_m - total_mapped:,} (off-cycle periods, pre-calendar years, interim instants)")
display(mc.filter("NOT (is_aligned AND (is_inst OR is_ytd OR is_disc))").groupBy("metric_period_type", "is_aligned", "is_ytd", "is_disc").count())

In [0]:
# Step 5: per concept and fiscal year -> reported YTD / discrete / instant values per quarter, then derive missing discrete quarters
S = F.struct("value", "accession_number", "filed_date", "period_start", "period_end", "form")
def pick(cond):
    return F.max_by(F.when(cond, S), F.when(cond, F.struct("filed_date", "accession_number")))

KEYS = ["cik", "metric", "concept", "priority", "unit", "metric_period_type", "is_additive", "fiscal_year"]
aggs = []
for n in range(1, 5):
    q = F.col("fiscal_quarter") == n
    aggs += [pick(q & F.col("is_ytd")).alias(f"ytd{n}"), pick(q & F.col("is_disc")).alias(f"disc{n}"), pick(q & F.col("is_inst")).alias(f"inst{n}")]
g = mapped.groupBy(*KEYS).agg(*aggs)

is_inst = F.col("metric_period_type") == "instant"
q_structs, cum = [], None
for n in range(1, 5):
    d, y, i = (F.col(f"disc{n}"), F.col(f"ytd{n}"), F.col(f"inst{n}"))
    dv, yv, iv = d.getField("value"), y.getField("value"), i.getField("value")
    if n == 1:
        dur_val = F.coalesce(dv, yv)
        dur_src = F.when(dv.isNotNull(), "reported_discrete").when(yv.isNotNull(), "reported_ytd")
        cum_n = F.coalesce(yv, dv)
    else:
        derived = F.when(F.col("is_additive"), yv - cum)
        dur_val = F.coalesce(dv, derived)
        dur_src = F.when(dv.isNotNull(), "reported_discrete").when(derived.isNotNull(), "derived_ytd_difference" if n < 4 else "derived_fy_minus_9m")
        cum_n = F.coalesce(yv, cum + dur_val)
    q_structs.append(F.struct(
        F.lit(n).alias("fiscal_quarter"),
        F.when(is_inst, iv).otherwise(dur_val).alias("value"),
        F.when(is_inst, F.when(iv.isNotNull(), "reported_instant")).otherwise(dur_src).alias("value_source"),
        F.when(is_inst, i.getField("accession_number")).otherwise(F.coalesce(d.getField("accession_number"), y.getField("accession_number"))).alias("source_accession"),
        F.when(is_inst, i.getField("filed_date")).otherwise(F.coalesce(d.getField("filed_date"), y.getField("filed_date"))).alias("source_filed_date"),
        F.when(is_inst, i.getField("period_end")).otherwise(F.coalesce(d.getField("period_end"), y.getField("period_end"))).alias("period_end")))
    cum = cum_n

q_by_concept = (g.select(*KEYS, F.explode(F.array(*q_structs)).alias("q")).select(*KEYS, "q.*").filter("value IS NOT NULL"))

# annual: reported FY value (YTD at Q4 / instant at FY end); fallback = sum of 4 reported discrete quarters
all_disc = F.col("disc1").isNotNull() & F.col("disc2").isNotNull() & F.col("disc3").isNotNull() & F.col("disc4").isNotNull()
sum_disc = sum(F.col(f"disc{n}.value") for n in range(1, 5))
ann_src_struct = F.when(is_inst, F.col("inst4")).otherwise(F.col("ytd4"))
a_by_concept = (g.select(*KEYS,
        F.coalesce(ann_src_struct.getField("value"), F.when(~is_inst & F.col("is_additive") & all_disc, sum_disc)).alias("value"),
        F.when(ann_src_struct.isNotNull(), "reported").when(~is_inst & F.col("is_additive") & all_disc, "derived_sum_of_quarters").alias("value_source"),
        ann_src_struct.getField("accession_number").alias("source_accession"), ann_src_struct.getField("filed_date").alias("source_filed_date"),
        ann_src_struct.getField("form").alias("source_form"),
        ann_src_struct.getField("period_start").alias("period_start"), ann_src_struct.getField("period_end").alias("period_end"))
    .filter("value IS NOT NULL"))

# concept resolution: highest-priority concept with a value wins per company-period; keep the losers count for transparency
def resolve(df, grain):
    w = Window.partitionBy(*grain).orderBy("priority")
    return (df.withColumn("n_candidate_concepts", F.count("*").over(Window.partitionBy(*grain)))
              .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn", "priority", "metric_period_type", "is_additive"))

annual = resolve(a_by_concept, ["cik", "metric", "fiscal_year"])
quarterly = resolve(q_by_concept, ["cik", "metric", "fiscal_year", "fiscal_quarter"])

# restatement tracking: first-ever reported value for the same concept/period vs the current (latest filed) value
orig = (fs_facts.filter("taxonomy = 'us-gaap'")
        .groupBy("cik", "concept", "unit", "period_start", "period_end")
        .agg(F.min_by("value", F.struct("filed_date", "accession_number")).alias("originally_reported_value"),
             F.min("filed_date").alias("original_filed_date"), F.countDistinct("accession_number").alias("n_filings_reporting")))
annual = (annual.alias("a").join(orig.alias("o"),
            (F.col("a.cik") == F.col("o.cik")) & (F.col("a.concept") == F.col("o.concept")) & (F.col("a.unit") == F.col("o.unit")) &
            F.col("a.period_start").eqNullSafe(F.col("o.period_start")) & (F.col("a.period_end") == F.col("o.period_end")), "left")
          .select("a.*", "o.originally_reported_value", "o.original_filed_date", "o.n_filings_reporting")
          .withColumn("is_restated", F.coalesce(F.col("originally_reported_value") != F.col("value"), F.lit(False))))

annual = stage(annual.select("cik", "fiscal_year", "metric", "value", "unit", "concept", "value_source", "source_accession", "source_form",
                             "source_filed_date", "period_start", "period_end", "originally_reported_value", "original_filed_date",
                             "n_filings_reporting", "is_restated", "n_candidate_concepts"),
               "fact_financials_annual", ["cik", "fiscal_year", "metric"], "Annual financial facts per company, fiscal year and metric (as currently reported) with lineage and restatement flag", ["cik", "metric"])
quarterly = stage(quarterly.select("cik", "fiscal_year", "fiscal_quarter", "metric", "value", "unit", "concept", "value_source",
                                   "source_accession", "source_filed_date", "period_end", "n_candidate_concepts"),
                  "fact_financials_quarterly", ["cik", "fiscal_year", "fiscal_quarter", "metric"], "Discrete quarterly financial facts per company and metric; derived values labelled in value_source", ["cik", "metric"])
print(f"annual rows={annual.count():,} quarterly rows={quarterly.count():,}")
display(quarterly.groupBy("value_source").count())
display(annual.groupBy("metric").agg(F.count("*").alias("rows"), F.count_if("is_restated").alias("restated"), F.countDistinct("concept").alias("concepts_used")).orderBy("metric"))

In [0]:
# Step 6: analyst-friendly wide tables with standard KPIs. Ratios use try_divide (no divide-by-zero failures).
def ratio(num, den):
    return F.round(F.expr(f"try_divide(cast(({num}) as double), cast(({den}) as double))"), 6)

def add_kpis(df):
    return (df.withColumn("gross_profit_calc", F.coalesce(F.col("gross_profit"), F.col("revenue") - F.col("cost_of_revenue")))
              .withColumn("free_cash_flow", F.col("operating_cash_flow") - F.col("capex"))
              .withColumn("gross_margin", ratio("gross_profit_calc", "revenue"))
              .withColumn("operating_margin", ratio("operating_income", "revenue"))
              .withColumn("net_margin", ratio("net_income", "revenue"))
              .withColumn("fcf_margin", ratio("free_cash_flow", "revenue"))
              .withColumn("rnd_intensity", ratio("rnd_expense", "revenue"))
              .withColumn("effective_tax_rate", ratio("income_tax_expense", "pretax_income")))

ann_w = annual.groupBy("cik", "fiscal_year").pivot("metric", METRIC_NAMES).agg(F.first("value"))
w_y = Window.partitionBy("cik").orderBy("fiscal_year")
def prev_year(col):  # only a true prior year counts (no gaps)
    return F.when(F.lag("fiscal_year").over(w_y) == F.col("fiscal_year") - 1, F.lag(col).over(w_y))
ann_w = (add_kpis(ann_w)
   .withColumn("prev_revenue", prev_year("revenue")).withColumn("prev_net_income", prev_year("net_income"))
   .withColumn("prev_eps_diluted", prev_year("eps_diluted")).withColumn("prev_total_equity", prev_year("total_equity"))
   .withColumn("prev_total_assets", prev_year("total_assets"))
   .withColumn("revenue_growth_yoy", ratio("revenue - prev_revenue", "prev_revenue"))
   .withColumn("net_income_growth_yoy", ratio("net_income - prev_net_income", "abs(prev_net_income)"))
   .withColumn("eps_diluted_growth_yoy", ratio("eps_diluted - prev_eps_diluted", "abs(prev_eps_diluted)"))
   .withColumn("roe", ratio("net_income", "(total_equity + prev_total_equity) / 2"))
   .withColumn("roa", ratio("net_income", "(total_assets + prev_total_assets) / 2"))
   .withColumn("debt_to_equity", ratio("long_term_debt", "total_equity"))
   .withColumn("current_ratio", ratio("current_assets", "current_liabilities"))
   .drop("prev_revenue", "prev_net_income", "prev_eps_diluted", "prev_total_equity", "prev_total_assets"))
ann_w = (ann_w.join(cal.select("cik", "fiscal_year", "fiscal_year_label", "fy_start", "fy_end"), ["cik", "fiscal_year"], "left")
              .join(dim_company.select("cik", "entity_name"), "cik", "left"))
ann_w = ann_w.select("cik", "entity_name", "fiscal_year", "fiscal_year_label", "fy_start", "fy_end", *[c for c in ann_w.columns if c not in ("cik", "entity_name", "fiscal_year", "fiscal_year_label", "fy_start", "fy_end")])
ann_w = stage(ann_w, "company_financials_annual", ["cik", "fiscal_year"], "Wide annual financial statements and KPIs per company and fiscal year", ["cik", "fiscal_year"])

qtr_w = (quarterly.groupBy("cik", "fiscal_year", "fiscal_quarter").pivot("metric", METRIC_NAMES).agg(F.first("value"))
         .join(quarterly.groupBy("cik", "fiscal_year", "fiscal_quarter").agg(F.max("period_end").alias("quarter_end")), ["cik", "fiscal_year", "fiscal_quarter"]))
w_q = Window.partitionBy("cik", "fiscal_quarter").orderBy("fiscal_year")
def same_q_prev_year(col):
    return F.when(F.lag("fiscal_year").over(w_q) == F.col("fiscal_year") - 1, F.lag(col).over(w_q))
qtr_w = qtr_w.withColumn("q_idx", F.col("fiscal_year") * 4 + F.col("fiscal_quarter"))
w_ttm = Window.partitionBy("cik").orderBy("q_idx").rangeBetween(-3, 0)
def ttm(col):  # trailing 12 months only when 4 consecutive quarters exist
    return F.when(F.count(col).over(w_ttm) == 4, F.sum(col).over(w_ttm))
qtr_w = (add_kpis(qtr_w).withColumn("prev_rev", same_q_prev_year("revenue")).withColumn("prev_ni", same_q_prev_year("net_income"))
   .withColumn("revenue_growth_yoy", ratio("revenue - prev_rev", "prev_rev"))
   .withColumn("net_income_growth_yoy", ratio("net_income - prev_ni", "abs(prev_ni)"))
   .withColumn("ttm_revenue", ttm("revenue")).withColumn("ttm_net_income", ttm("net_income"))
   .withColumn("ttm_operating_cash_flow", ttm("operating_cash_flow")).withColumn("ttm_free_cash_flow", ttm("free_cash_flow"))
   .withColumn("ttm_net_margin", ratio("ttm_net_income", "ttm_revenue"))
   .drop("prev_rev", "prev_ni", "q_idx")
   .join(dim_company.select("cik", "entity_name"), "cik", "left")
   .withColumn("fiscal_period_label", F.concat(F.lit("FY"), F.col("fiscal_year").cast("string"), F.lit("-Q"), F.col("fiscal_quarter").cast("string"))))
lead = ["cik", "entity_name", "fiscal_year", "fiscal_quarter", "fiscal_period_label", "quarter_end"]
qtr_w = qtr_w.select(*lead, *[c for c in qtr_w.columns if c not in lead])
qtr_w = stage(qtr_w, "company_financials_quarterly", ["cik", "fiscal_year", "fiscal_quarter"], "Wide discrete-quarter financials and KPIs incl. TTM per company", ["cik", "fiscal_year"])
display(ann_w.filter("fiscal_year >= 2022").select("entity_name", "fiscal_year_label", "revenue", "net_income", "eps_diluted", "gross_margin", "operating_margin", "net_margin", "free_cash_flow", "revenue_growth_yoy", "roe").orderBy("entity_name", "fiscal_year"))

In [0]:
# Step 7: ERROR checks block publishing; WARN checks are recorded for review
results = []
def check(name, severity, failed_df, checked, details=""):
    n = failed_df.count()
    results.append((RUN_ID, name, severity, int(n), int(checked), n == 0, details))
    return failed_df

# 1. grain uniqueness of every staged table
for name, meta in STAGED.items():
    t = spark.table(meta["stg"])
    check(f"unique_grain:{name}", "ERROR", t.groupBy(*meta["keys"]).count().filter("count > 1"), t.count(), ",".join(meta["keys"]))

# 2. known published figures (10-K / earnings releases, USD) - proves concept choice, fiscal-year mapping and Q4 derivation
KNOWN = [(320193, 2024, 0, "revenue", 391035000000), (320193, 2024, 0, "net_income", 93736000000),
         (789019, 2024, 0, "revenue", 245122000000), (1018724, 2024, 0, "net_income", 59248000000),
         (1045810, 2024, 0, "revenue", 60922000000), (19617, 2024, 0, "net_income", 58471000000),
         (1318605, 2024, 0, "revenue", 97690000000), (104169, 2024, 0, "revenue", 648125000000),
         (320193, 2025, 1, "revenue", 124300000000), (320193, 2024, 4, "revenue", 94930000000)]
known = spark.createDataFrame(KNOWN, "cik long, fiscal_year int, fiscal_quarter int, metric string, expected double")
got = (annual.select("cik", "fiscal_year", F.lit(0).alias("fiscal_quarter"), "metric", "value")
       .unionByName(quarterly.select("cik", "fiscal_year", "fiscal_quarter", "metric", "value")))
kn = known.join(got, ["cik", "fiscal_year", "fiscal_quarter", "metric"], "left")
check("known_published_figures", "ERROR", kn.filter("value IS NULL OR abs(cast(value as double) - expected) > 1"), len(KNOWN))

# 3. fiscal calendar must be contiguous (next start = previous end + 1 day)
w_c = Window.partitionBy("cik").orderBy("fiscal_year")
gaps = cal.withColumn("prev_end", F.lag("fy_end").over(w_c)).filter("prev_end IS NOT NULL AND datediff(fy_start, prev_end) <> 1")
check("fiscal_calendar_contiguous", "ERROR", gaps, cal.count())

# 4. revenue must not be negative
check("revenue_non_negative", "ERROR", annual.filter("metric = 'revenue' AND value < 0").unionByName(
      quarterly.filter("metric = 'revenue' AND value < 0").drop("fiscal_quarter"), allowMissingColumns=True), annual.count())

# 5. balance sheet identity: assets = liabilities + equity (0.5% tolerance)
bs = ann_w.filter("total_assets IS NOT NULL AND liabilities_and_equity IS NOT NULL")
check("balance_sheet_identity", "WARN", bs.filter("abs(total_assets - liabilities_and_equity) > 0.005 * abs(total_assets)"), bs.count())

# 6. four discrete quarters must sum to the annual value (additive metrics, 0.5% tolerance)
add_metrics = [m[0] for m in METRICS if m[5]]
qs = (quarterly.filter(F.col("metric").isin(add_metrics)).groupBy("cik", "fiscal_year", "metric")
      .agg(F.count("*").alias("nq"), F.sum("value").alias("q_sum")).filter("nq = 4")
      .join(annual.select("cik", "fiscal_year", "metric", "value"), ["cik", "fiscal_year", "metric"]))
q_bad = check("quarters_sum_to_annual", "WARN", qs.filter("abs(q_sum - value) > 0.005 * abs(value)"), qs.count())

# 7. reported gross profit consistent with revenue - cost of revenue (1%)
gp = ann_w.filter("gross_profit IS NOT NULL AND revenue IS NOT NULL AND cost_of_revenue IS NOT NULL")
check("gross_profit_consistency", "WARN", gp.filter("abs(gross_profit - (revenue - cost_of_revenue)) > 0.01 * abs(revenue)"), gp.count())

# 8. core metric coverage for closed fiscal years since 2010
core = ["revenue", "net_income", "eps_diluted", "total_assets", "operating_cash_flow"]
cov = cal.filter("is_closed AND fiscal_year >= 2010").select("cik", "fiscal_year").join(ann_w, ["cik", "fiscal_year"], "left")
cov_gaps = cov.filter(" OR ".join(f"{c} IS NULL" for c in core))
check("core_metric_coverage", "WARN", cov_gaps, cov.count(), ",".join(core))

# 9. implausible margins
check("net_margin_plausible", "WARN", ann_w.filter("abs(net_margin) > 1"), ann_w.count())

dq_df = spark.createDataFrame(results, "run_id string, check_name string, severity string, failed_rows bigint, checked_rows bigint, passed boolean, details string").withColumn("checked_at", F.current_timestamp())
dq_df.write.mode("append").saveAsTable(f"{GLD}.dq_results")
DQ_ERRORS = sum(1 for r in results if r[2] == "ERROR" and not r[5])
DQ_WARNS  = sum(1 for r in results if r[2] == "WARN" and not r[5])
display(dq_df.drop("run_id", "checked_at"))
display(kn.withColumn("diff", F.col("value").cast("double") - F.col("expected")))
if DQ_WARNS:
    display(cov_gaps.select("cik", "entity_name", "fiscal_year", *core).orderBy("cik", "fiscal_year"))

In [0]:
# Step 8: publish only when all ERROR checks pass. CREATE OR REPLACE is atomic and keeps Delta history (rollback via RESTORE).
if DQ_ERRORS:
    fail_run(f"{DQ_ERRORS} ERROR-level DQ checks failed - gold NOT published; see {GLD}.dq_results run_id={RUN_ID}")

counts = {}
for name, meta in STAGED.items():
    spark.sql(f"""CREATE OR REPLACE TABLE {GLD}.{name} CLUSTER BY ({', '.join(meta['cluster_by'])})
                  COMMENT '{meta['comment']}' TBLPROPERTIES ('quality' = 'gold', 'source_silver_version' = '{SILVER_VERSION}', 'gold_run_id' = '{RUN_ID}')
                  AS SELECT * FROM {meta['stg']}""")
    counts[name] = spark.table(f"{GLD}.{name}").count()
    spark.sql(f"DROP TABLE IF EXISTS {meta['stg']}")

map_sql = "map(" + ", ".join(f"'{k}', {v}L" for k, v in counts.items()) + ")"
spark.sql(f"""UPDATE {GLD}.audit_runs SET run_end = current_timestamp(), status = '{'SUCCESS' if DQ_WARNS == 0 else 'SUCCESS_WITH_WARNINGS'}',
              table_row_counts = {map_sql}, dq_errors = {DQ_ERRORS}, dq_warnings = {DQ_WARNS} WHERE run_id = '{RUN_ID}'""")
display(spark.table(f"{GLD}.audit_runs").filter(F.col("run_id") == RUN_ID))
display(spark.table(f"{GLD}.company_financials_quarterly").filter("cik = 320193 AND fiscal_year >= 2024")
        .select("entity_name", "fiscal_period_label", "quarter_end", "revenue", "net_income", "eps_diluted", "operating_cash_flow", "gross_margin", "ttm_revenue", "revenue_growth_yoy")
        .orderBy("fiscal_year", "fiscal_quarter"))

## Business questions answered from gold

**Analytical assumptions (BI model)**
* Grain: `company_financials_annual` = company x fiscal year; `company_financials_quarterly` = company x fiscal quarter (discrete). Facts in `fact_financials_*` carry lineage (concept, accession, value_source).
* "Latest year" = each company's **last closed fiscal year** (`dim_company.last_closed_fiscal_year`), because year-ends differ (Apple Sep, Microsoft Jun, Walmart/NVIDIA Jan, others Dec). Comparisons are fiscal-year aligned, not calendar aligned.
* Values are **as currently reported** (latest 10-K/10-Q incl. restatements and stock-split adjusted per-share data).
* JPMorgan is a bank: revenue = net revenue after interest expense, there is no gross margin / current ratio, and operating cash flow is driven by trading balances, so cash-flow and liquidity KPIs are not comparable for JPM and are flagged.
* Growth uses compound annual growth rate (CAGR) over 5 fiscal years; CAGR is undefined when the start or end value is <= 0.

**Questions**
1. Who is the largest and most profitable company in its latest fiscal year?
2. Who grew fastest over the last 5 fiscal years (revenue, net income, EPS CAGR)?
3. How have operating margins trended over the last 6 fiscal years?
4. How well do profits convert into free cash flow?
5. Who generates the best returns on equity and assets?
6. Whose balance sheet is strongest (net cash, leverage, liquidity)?
7. How much of earnings is returned as dividends?
8. What is the latest quarterly momentum (YoY growth, trailing 12 months)?
9. Who invests most in R&D relative to revenue, and is it rising?
10. How often are reported annual figures restated, and by how much?

In [0]:
%sql
-- Q1: Who is the largest and most profitable company in its latest closed fiscal year?
SELECT a.entity_name, a.fiscal_year_label, a.fy_end,
       round(a.revenue / 1e9, 2)          AS revenue_bn,
       round(a.net_income / 1e9, 2)       AS net_income_bn,
       round(a.gross_margin * 100, 1)     AS gross_margin_pct,
       round(a.operating_margin * 100, 1) AS operating_margin_pct,
       round(a.net_margin * 100, 1)       AS net_margin_pct,
       rank() OVER (ORDER BY a.revenue DESC)    AS revenue_rank,
       rank() OVER (ORDER BY a.net_income DESC) AS profit_rank
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual') AS a
JOIN IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company') AS d
  ON a.cik = d.cik AND a.fiscal_year = d.last_closed_fiscal_year
ORDER BY revenue_rank

In [0]:
%sql
-- Q2: Who grew fastest over the last 5 fiscal years? CAGR only when start and end values are positive.
WITH a AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual')),
     d AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company'))
SELECT d.entity_name,
       concat(s.fiscal_year_label, ' -> ', e.fiscal_year_label) AS window,
       round(s.revenue / 1e9, 1) AS revenue_start_bn, round(e.revenue / 1e9, 1) AS revenue_end_bn,
       round((pow(try_divide(cast(e.revenue AS DOUBLE), cast(s.revenue AS DOUBLE)), 0.2) - 1) * 100, 1) AS revenue_cagr_pct,
       CASE WHEN s.net_income > 0 AND e.net_income > 0
            THEN round((pow(cast(e.net_income AS DOUBLE) / cast(s.net_income AS DOUBLE), 0.2) - 1) * 100, 1) END AS net_income_cagr_pct,
       CASE WHEN s.eps_diluted > 0 AND e.eps_diluted > 0
            THEN round((pow(cast(e.eps_diluted AS DOUBLE) / cast(s.eps_diluted AS DOUBLE), 0.2) - 1) * 100, 1) END AS eps_diluted_cagr_pct
FROM d
JOIN a e ON e.cik = d.cik AND e.fiscal_year = d.last_closed_fiscal_year
JOIN a s ON s.cik = d.cik AND s.fiscal_year = d.last_closed_fiscal_year - 5
ORDER BY revenue_cagr_pct DESC

In [0]:
%sql
-- Q3: Operating margin (%) over the last 6 fiscal years, relative to each company's latest closed year (Y0)
WITH t AS (
  SELECT a.entity_name, a.fiscal_year - d.last_closed_fiscal_year AS rel_year, round(a.operating_margin * 100, 1) AS op_margin_pct
  FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual') AS a
  JOIN IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company') AS d ON a.cik = d.cik
  WHERE a.fiscal_year BETWEEN d.last_closed_fiscal_year - 5 AND d.last_closed_fiscal_year)
SELECT *, round(`Y0` - `Y-5`, 1) AS change_pts
FROM t PIVOT (first(op_margin_pct) FOR rel_year IN (-5 AS `Y-5`, -4 AS `Y-4`, -3 AS `Y-3`, -2 AS `Y-2`, -1 AS `Y-1`, 0 AS `Y0`))
ORDER BY `Y0` DESC NULLS LAST

In [0]:
%sql
-- Q4: How well does profit convert into free cash flow (FCF = operating cash flow - capex)?
SELECT a.entity_name, a.fiscal_year_label,
       round(a.net_income / 1e9, 2)          AS net_income_bn,
       round(a.operating_cash_flow / 1e9, 2) AS operating_cash_flow_bn,
       round(a.capex / 1e9, 2)               AS capex_bn,
       round(a.free_cash_flow / 1e9, 2)      AS free_cash_flow_bn,
       round(try_divide(a.free_cash_flow, a.net_income), 2) AS fcf_conversion,
       round(a.fcf_margin * 100, 1)          AS fcf_margin_pct,
       round(try_divide(a.capex, a.revenue) * 100, 1) AS capex_intensity_pct,
       CASE WHEN a.cik = 19617 THEN 'bank - not comparable' END AS note
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual') AS a
JOIN IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company') AS d
  ON a.cik = d.cik AND a.fiscal_year = d.last_closed_fiscal_year
ORDER BY (a.cik = 19617), fcf_conversion DESC

In [0]:
%sql
-- Q5: Returns on average equity / assets in the latest closed fiscal year (and 5-year average ROE)
WITH a AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual')),
     d AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company'))
SELECT d.entity_name,
       max_by(a.fiscal_year_label, a.fiscal_year)            AS latest_fy,
       round(max_by(a.roe, a.fiscal_year) * 100, 1)         AS roe_pct,
       round(max_by(a.roa, a.fiscal_year) * 100, 1)         AS roa_pct,
       round(avg(a.roe) * 100, 1)                           AS roe_5y_avg_pct,
       round(max_by(try_divide(a.total_assets, a.total_equity), a.fiscal_year), 1) AS equity_multiplier
FROM d JOIN a ON a.cik = d.cik AND a.fiscal_year BETWEEN d.last_closed_fiscal_year - 4 AND d.last_closed_fiscal_year
GROUP BY d.entity_name
ORDER BY roe_pct DESC

In [0]:
%sql
-- Q6: Balance-sheet strength at the latest fiscal year end
SELECT a.entity_name, a.fiscal_year_label, a.fy_end,
       round(a.cash / 1e9, 1)                    AS cash_bn,
       round(a.long_term_debt / 1e9, 1)          AS long_term_debt_bn,
       round((a.cash - a.long_term_debt) / 1e9, 1) AS net_cash_bn,
       round(a.debt_to_equity, 2)                AS lt_debt_to_equity,
       round(a.current_ratio, 2)                 AS current_ratio,
       round(try_divide(a.total_equity, a.total_assets) * 100, 1) AS equity_to_assets_pct,
       CASE WHEN a.cik = 19617 THEN 'bank - deposits not in debt, no current ratio' END AS note
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual') AS a
JOIN IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company') AS d
  ON a.cik = d.cik AND a.fiscal_year = d.last_closed_fiscal_year
ORDER BY net_cash_bn DESC NULLS LAST

In [0]:
%sql
-- Q7: How much of earnings is returned as dividends? (dividends declared per share / diluted EPS)
WITH a AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual')),
     d AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company'))
SELECT d.entity_name, e.fiscal_year_label,
       e.eps_diluted, e.dividends_per_share,
       round(try_divide(e.dividends_per_share, e.eps_diluted) * 100, 1) AS payout_ratio_pct,
       s.dividends_per_share AS dps_5y_ago,
       CASE WHEN s.dividends_per_share > 0 AND e.dividends_per_share > 0
            THEN round((pow(cast(e.dividends_per_share AS DOUBLE) / cast(s.dividends_per_share AS DOUBLE), 0.2) - 1) * 100, 1) END AS dps_cagr_5y_pct,
       CASE WHEN e.dividends_per_share IS NULL THEN 'no dividend reported' END AS note
FROM d
JOIN a e ON e.cik = d.cik AND e.fiscal_year = d.last_closed_fiscal_year
LEFT JOIN a s ON s.cik = d.cik AND s.fiscal_year = d.last_closed_fiscal_year - 5
ORDER BY payout_ratio_pct DESC NULLS LAST

In [0]:
%sql
-- Q8: Latest reported quarter per company: YoY growth vs the same fiscal quarter last year, plus trailing-12-month view
WITH q AS (
  SELECT *, row_number() OVER (PARTITION BY cik ORDER BY fiscal_year DESC, fiscal_quarter DESC) AS rn
  FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_quarterly')
  WHERE revenue IS NOT NULL)
SELECT entity_name, fiscal_period_label, quarter_end,
       round(revenue / 1e9, 2)                AS revenue_bn,
       round(revenue_growth_yoy * 100, 1)     AS revenue_yoy_pct,
       round(net_income / 1e9, 2)             AS net_income_bn,
       round(net_income_growth_yoy * 100, 1)  AS net_income_yoy_pct,
       round(ttm_revenue / 1e9, 1)            AS ttm_revenue_bn,
       round(ttm_net_margin * 100, 1)         AS ttm_net_margin_pct
FROM q WHERE rn = 1
ORDER BY revenue_yoy_pct DESC NULLS LAST

In [0]:
%sql
-- Q9: R&D as % of revenue, latest closed fiscal year vs 5 years earlier (companies that report R&D)
WITH a AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.company_financials_annual')),
     d AS (SELECT * FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company'))
SELECT d.entity_name, e.fiscal_year_label,
       round(e.rnd_expense / 1e9, 1)     AS rnd_bn,
       round(e.rnd_intensity * 100, 1)   AS rnd_pct_revenue,
       round(s.rnd_intensity * 100, 1)   AS rnd_pct_revenue_5y_ago,
       round((e.rnd_intensity - s.rnd_intensity) * 100, 1) AS change_pts
FROM d
JOIN a e ON e.cik = d.cik AND e.fiscal_year = d.last_closed_fiscal_year
LEFT JOIN a s ON s.cik = d.cik AND s.fiscal_year = d.last_closed_fiscal_year - 5
WHERE e.rnd_expense IS NOT NULL
ORDER BY rnd_pct_revenue DESC

In [0]:
%sql
-- Q10: How often are annual figures restated (current value differs from the first filed value) and by how much?
WITH f AS (
  SELECT f.*, d.entity_name,
         try_divide(cast(f.value - f.originally_reported_value AS DOUBLE), abs(cast(f.originally_reported_value AS DOUBLE))) AS pct_change
  FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.fact_financials_annual') AS f
  JOIN IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_company') AS d ON f.cik = d.cik
  WHERE f.value_source = 'reported')
SELECT entity_name,
       count(*)                                        AS annual_facts,
       count_if(is_restated)                           AS restated_facts,
       round(count_if(is_restated) / count(*) * 100, 1) AS restated_pct,
       count_if(is_restated AND abs(pct_change) >= 0.01) AS material_restatements_1pct,
       max_by(concat(metric, ' FY', fiscal_year), abs(pct_change)) FILTER (WHERE is_restated) AS largest_restatement,
       round(max(abs(pct_change)) FILTER (WHERE is_restated) * 100, 1) AS largest_change_pct
FROM f
GROUP BY entity_name
ORDER BY restated_pct DESC